# Ray on AKS with Blobfuse2 + Anyscale Operator
## End-to-End Validation & Deployment Notebook

This notebook validates your entire setup:
1. **Azure Authentication & Subscription**
2. **AKS Cluster Connectivity**
3. **Storage Account & Blob Container**
4. **Blobfuse2 StorageClasses**
5. **PVCs (Dataset + Checkpoints)**
6. **Anyscale Operator Health**
7. **Data Availability**
8. **Deploy Ray Job**

---
## 0. Configuration
Load environment variables from `.env.example`.

In [ ]:
import subprocess, os, json, sys, time

def run(cmd, capture=True, shell=True):
    """Run a shell command and return (returncode, stdout, stderr)."""
    r = subprocess.run(cmd, capture_output=capture, text=True, shell=shell)
    return r.returncode, r.stdout.strip(), r.stderr.strip()

def ok(msg):   print(f"\u2705 {msg}")
def warn(msg): print(f"\u26A0\uFE0F  {msg}")
def fail(msg): print(f"\u274C {msg}")

# Load .env.example
env = {}
env_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '.env.example')
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, val = line.split('=', 1)
                env[key.strip()] = val.strip().strip('"')
    ok(f"Loaded {len(env)} variables from .env.example")
else:
    fail("No .env.example found!")

# Key variables
SUBSCRIPTION   = env.get('SUBSCRIPTION', '')
RG             = env.get('RG', '')
LOC            = env.get('LOC', '')
AKS            = env.get('AKS', '')
SA             = env.get('SA', '')
CONTAINER      = env.get('CONTAINER', '')
DATA_DIR       = env.get('DATA_DIR', '/mnt/blob/datasets')
CHECKPOINT_DIR = env.get('CHECKPOINT_DIR', '/mnt/blob/checkpoints')
APP_DIR        = env.get('APP_DIR', '/app')
CACHE_DIR      = env.get('CACHE_DIR', '/mnt/blobfusecache')
CHECKPOINT_CACHE = env.get('CHECKPOINT_CACHE', '/mnt/blobfusecache_ckpt')
NUM_WORKERS    = env.get('NUM_WORKERS', '20')
WORKER_REPLICAS = env.get('WORKER_REPLICAS', '20')
STORAGE_KEY    = env.get('STORAGE_ACCOUNT_KEY', '')

print(f"\n--- Configuration ---")
print(f"Subscription : {SUBSCRIPTION}")
print(f"Resource Group: {RG}")
print(f"Location     : {LOC}")
print(f"AKS Cluster  : {AKS}")
print(f"Storage Acct : {SA}")
print(f"Container    : {CONTAINER}")
print(f"Data Dir     : {DATA_DIR}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"Workers      : {WORKER_REPLICAS} replicas, {NUM_WORKERS} workers")

---
## 1. Azure Authentication

In [ ]:
# Check Azure CLI login
rc, out, err = run('az account show -o json')
if rc == 0:
    acct = json.loads(out)
    ok(f"Logged in as: {acct.get('user', {}).get('name', 'N/A')}")
    ok(f"Subscription: {acct.get('name', 'N/A')} ({acct.get('id', 'N/A')})")
    if acct.get('id') != SUBSCRIPTION:
        warn(f"Active subscription doesn't match .env.example ({SUBSCRIPTION})")
        print("  Setting correct subscription...")
        run(f'az account set --subscription {SUBSCRIPTION}')
        ok("Subscription switched.")
else:
    fail("Not logged in to Azure CLI. Run: az login")

---
## 2. AKS Cluster Connectivity

In [ ]:
# Get AKS credentials and verify kubectl connectivity
rc, out, err = run(f'az aks get-credentials -g {RG} -n {AKS} --overwrite-existing')
if rc == 0:
    ok(f"Kubeconfig fetched for cluster '{AKS}'")
else:
    fail(f"Failed to get AKS credentials: {err}")

# Verify cluster access
rc, out, err = run('kubectl cluster-info')
if rc == 0:
    ok("kubectl connected to cluster")
    for line in out.split('\n')[:2]:
        print(f"  {line}")
else:
    fail(f"Cannot reach cluster: {err}")

# Node status
rc, out, err = run('kubectl get nodes -o wide')
if rc == 0:
    lines = out.strip().split('\n')
    ready_count = sum(1 for l in lines[1:] if 'Ready' in l)
    total_count = len(lines) - 1
    ok(f"{ready_count}/{total_count} nodes are Ready")
    print(out)

---
## 3. Storage Account & Blob Container

In [ ]:
# Verify storage account exists
rc, out, err = run(f'az storage account show -g {RG} -n {SA} --query "{{name:name, location:location, sku:sku.name, kind:kind, provisioningState:provisioningState}}" -o json')
if rc == 0:
    sa_info = json.loads(out)
    ok(f"Storage Account '{SA}' found")
    print(f"  Location : {sa_info.get('location')}")
    print(f"  SKU      : {sa_info.get('sku')}")
    print(f"  Kind     : {sa_info.get('kind')}")
    print(f"  State    : {sa_info.get('provisioningState')}")
else:
    fail(f"Storage account '{SA}' not found in RG '{RG}': {err}")

# Verify blob container exists
rc, out, err = run(f'az storage container exists -n {CONTAINER} --account-name {SA} --query exists -o tsv')
if rc == 0 and out.lower() == 'true':
    ok(f"Blob container '{CONTAINER}' exists")
else:
    fail(f"Blob container '{CONTAINER}' not found")

# Check blob CSI driver is enabled on AKS
rc, out, err = run(f'az aks show -g {RG} -n {AKS} --query "storageProfile.blobCsiDriver.enabled" -o tsv')
if rc == 0 and out.lower() == 'true':
    ok("Blob CSI driver is ENABLED on AKS")
else:
    warn("Blob CSI driver may not be enabled. Run: az aks update -g {RG} -n {AKS} --enable-blob-driver")

---
## 4. StorageClasses (Blobfuse2)

In [ ]:
expected_scs = ['blobfuse-training-data', 'blobfuse-checkpoints']

rc, out, err = run('kubectl get storageclass -o json')
if rc == 0:
    scs = json.loads(out)
    existing_scs = {item['metadata']['name']: item for item in scs.get('items', [])}
    
    for sc_name in expected_scs:
        if sc_name in existing_scs:
            sc = existing_scs[sc_name]
            provisioner = sc.get('provisioner', 'N/A')
            params = sc.get('parameters', {})
            ok(f"StorageClass '{sc_name}' exists")
            print(f"  Provisioner     : {provisioner}")
            print(f"  Storage Account : {params.get('storageAccount', 'N/A')}")
            print(f"  Container       : {params.get('containername', 'N/A')}")
            print(f"  Resource Group  : {params.get('resourceGroup', 'N/A')}")
            print(f"  WorkloadIdentity: {params.get('mountWithWorkloadIdentityToken', 'N/A')}")
            mount_opts = sc.get('mountOptions', [])
            if mount_opts:
                print(f"  Mount Options   : {', '.join(mount_opts[:5])}{'...' if len(mount_opts)>5 else ''}")
            
            # Validate storage account matches .env
            if params.get('storageAccount') and params['storageAccount'] != SA and '__' not in params['storageAccount']:
                warn(f"  StorageClass '{sc_name}' uses '{params['storageAccount']}' but .env has '{SA}'")
            elif '__' in params.get('storageAccount', ''):
                fail(f"  StorageClass '{sc_name}' still has placeholder __STORAGE_ACCOUNT__ — not deployed yet")
        else:
            fail(f"StorageClass '{sc_name}' NOT FOUND — run quick-deploy.ps1 to create it")
else:
    fail(f"Could not list StorageClasses: {err}")

---
## 5. Persistent Volume Claims (PVCs)
Verify the dataset and checkpoint PVCs are created and bound.

In [ ]:
expected_pvcs = {
    'blob-pvc-dataset': {
        'storageClass': 'blobfuse-training-data',
        'accessMode': 'ReadOnlyMany',
        'description': 'AG News dataset (read-only)'
    },
    'blob-pvc-checkpoint': {
        'storageClass': 'blobfuse-checkpoints',
        'accessMode': 'ReadWriteMany',
        'description': 'Model checkpoints (read-write)'
    }
}

rc, out, err = run('kubectl get pvc -o json')
if rc == 0:
    pvcs = json.loads(out)
    existing_pvcs = {item['metadata']['name']: item for item in pvcs.get('items', [])}
    
    for pvc_name, expected in expected_pvcs.items():
        if pvc_name in existing_pvcs:
            pvc = existing_pvcs[pvc_name]
            status = pvc.get('status', {}).get('phase', 'Unknown')
            sc = pvc['spec'].get('storageClassName', 'N/A')
            access = pvc['spec'].get('accessModes', [])
            capacity = pvc['spec'].get('resources', {}).get('requests', {}).get('storage', 'N/A')
            
            if status == 'Bound':
                ok(f"PVC '{pvc_name}' is BOUND ({expected['description']})")
            elif status == 'Pending':
                warn(f"PVC '{pvc_name}' is PENDING — may bind on first pod mount")
            else:
                fail(f"PVC '{pvc_name}' status: {status}")
            
            print(f"  StorageClass : {sc}")
            print(f"  Access Modes : {', '.join(access)}")
            print(f"  Capacity     : {capacity}")
            
            # Validate StorageClass matches
            if sc != expected['storageClass']:
                warn(f"  Expected StorageClass '{expected['storageClass']}' but got '{sc}'")
            else:
                ok(f"  StorageClass matches expected: {sc}")
            
            # Validate access mode
            if expected['accessMode'] in access:
                ok(f"  Access mode '{expected['accessMode']}' is correct")
            else:
                warn(f"  Expected access mode '{expected['accessMode']}' not found in {access}")
        else:
            fail(f"PVC '{pvc_name}' NOT FOUND — run quick-deploy.ps1")
else:
    fail(f"Could not list PVCs: {err}")

# Show all PVCs
print("\n--- All PVCs ---")
rc, out, _ = run('kubectl get pvc -o wide')
if rc == 0:
    print(out)

---
## 6. Anyscale Operator Health

In [ ]:
# Check Anyscale operator namespace and pods
rc, out, err = run('kubectl get namespace anyscale-operator -o json 2>nul')
if rc == 0:
    ns = json.loads(out)
    phase = ns.get('status', {}).get('phase', 'Unknown')
    ok(f"Namespace 'anyscale-operator' exists (phase: {phase})")
else:
    warn("Namespace 'anyscale-operator' not found — checking default namespace")

# Check operator deployment
for ns in ['anyscale-operator', 'default']:
    rc, out, err = run(f'kubectl get deployments -n {ns} -o json 2>nul')
    if rc == 0:
        deps = json.loads(out)
        for dep in deps.get('items', []):
            name = dep['metadata']['name']
            ready = dep.get('status', {}).get('readyReplicas', 0)
            desired = dep['spec'].get('replicas', 0)
            if 'anyscale' in name.lower() or 'operator' in name.lower():
                if ready == desired and ready > 0:
                    ok(f"Deployment '{name}' in '{ns}': {ready}/{desired} ready")
                else:
                    warn(f"Deployment '{name}' in '{ns}': {ready}/{desired} ready")

# List operator pods
print("\n--- Anyscale Operator Pods ---")
rc, out, _ = run('kubectl get pods -n anyscale-operator -o wide 2>nul')
if rc == 0 and out:
    print(out)
else:
    rc, out, _ = run('kubectl get pods --all-namespaces -o wide | findstr anyscale')
    if out:
        print(out)
    else:
        warn("No Anyscale operator pods found")

# Helm releases
print("\n--- Helm Releases (Anyscale) ---")
rc, out, _ = run('helm list -A -o json')
if rc == 0:
    releases = json.loads(out)
    anyscale_releases = [r for r in releases if 'anyscale' in r.get('name', '').lower()]
    if anyscale_releases:
        for r in anyscale_releases:
            status = r.get('status', 'unknown')
            icon = '\u2705' if status == 'deployed' else '\u26A0\uFE0F'
            print(f"  {icon} {r['name']} (ns: {r['namespace']}) — chart: {r.get('chart','')} — status: {status}")
    else:
        warn("No Anyscale Helm releases found")

---
## 7. Ingress & DNS Connectivity

In [ ]:
# Check ingress-nginx
print("--- Ingress Controller ---")
rc, out, _ = run('kubectl get pods -n ingress-nginx -o wide 2>nul')
if rc == 0 and out and 'nginx' in out.lower():
    ok("Ingress-nginx controller found")
    print(out)
else:
    # Try all namespaces
    rc, out, _ = run('kubectl get pods --all-namespaces | findstr nginx')
    if out:
        ok("Nginx ingress pods found")
        print(out)
    else:
        warn("No ingress-nginx pods found")

# Check LoadBalancer service IPs
print("\n--- LoadBalancer Services ---")
rc, out, _ = run('kubectl get svc --all-namespaces -o json')
if rc == 0:
    svcs = json.loads(out)
    for svc in svcs.get('items', []):
        if svc['spec'].get('type') == 'LoadBalancer':
            name = svc['metadata']['name']
            ns = svc['metadata']['namespace']
            ingress = svc.get('status', {}).get('loadBalancer', {}).get('ingress', [])
            ip = ingress[0].get('ip', 'Pending') if ingress else 'Pending'
            if ip != 'Pending':
                ok(f"{ns}/{name} -> {ip}")
            else:
                warn(f"{ns}/{name} -> IP is Pending")

---
## 8. Dataset Availability in Blob Storage

In [ ]:
# Check if AG News dataset exists in blob storage
rc, out, err = run(f'az storage blob list --account-name {SA} --container-name {CONTAINER} --prefix "ag_news" --query "[].{{name:name, size:properties.contentLength}}" -o json --num-results 20')
if rc == 0:
    blobs = json.loads(out)
    if blobs:
        total_size = sum(b.get('size', 0) for b in blobs)
        ok(f"Dataset found: {len(blobs)} blobs under 'ag_news/' ({total_size / (1024*1024):.1f} MB)")
        for b in blobs[:10]:
            print(f"  {b['name']} ({b.get('size', 0) / 1024:.1f} KB)")
        if len(blobs) > 10:
            print(f"  ... and {len(blobs) - 10} more")
    else:
        warn(f"No blobs found under 'ag_news/' prefix in container '{CONTAINER}'")
        print("  Run: python data/prepare_ag_news.py && az storage blob upload-batch ...")
else:
    fail(f"Could not list blobs: {err}")

# Also check for checkpoints directory
rc, out, err = run(f'az storage blob list --account-name {SA} --container-name {CONTAINER} --prefix "checkpoints" --query "length(@)" -o tsv --num-results 5')
if rc == 0:
    count = int(out) if out.strip().isdigit() else 0
    if count > 0:
        ok(f"Checkpoints directory exists ({count} blobs found)")
    else:
        ok("Checkpoints directory is empty (expected before first training run)")

---
## 9. AKS Managed Identity → Storage Role Assignments
Verify the AKS kubelet identity has `Storage Blob Data Contributor` on the storage account.

In [ ]:
# Get AKS kubelet identity
rc, out, _ = run(f'az aks show -g {RG} -n {AKS} --query "identityProfile.kubeletidentity.objectId" -o tsv')
kubelet_oid = out.strip() if rc == 0 else ''

if kubelet_oid:
    ok(f"AKS kubelet identity OID: {kubelet_oid}")
    
    # Get storage account resource ID
    rc, sa_id, _ = run(f'az storage account show -g {RG} -n {SA} --query id -o tsv')
    
    if rc == 0 and sa_id:
        # Check role assignments
        rc, out, _ = run(f'az role assignment list --assignee {kubelet_oid} --scope {sa_id.strip()} -o json')
        if rc == 0:
            roles = json.loads(out)
            role_names = [r.get('roleDefinitionName', '') for r in roles]
            
            required_roles = ['Storage Blob Data Contributor', 'Storage Blob Data Owner']
            has_blob_role = any(r in role_names for r in required_roles)
            
            if has_blob_role:
                ok(f"Kubelet has storage role: {[r for r in role_names if 'Storage' in r]}")
            else:
                warn(f"Kubelet roles on storage: {role_names}")
                warn("Missing 'Storage Blob Data Contributor' — blobfuse mounts may fail!")
                print(f"  Fix: az role assignment create --assignee {kubelet_oid} --role 'Storage Blob Data Contributor' --scope {sa_id.strip()}")
        else:
            warn("Could not check role assignments")
    else:
        warn("Could not get storage account resource ID")
else:
    warn("Could not find AKS kubelet identity")

---
## 10. End-to-End Mount Test
Spin up a quick test pod to verify blob storage mounts correctly via the PVCs.

In [ ]:
# Create a test pod that mounts both PVCs and lists files
test_pod_yaml = f"""
apiVersion: v1
kind: Pod
metadata:
  name: pvc-mount-test
spec:
  restartPolicy: Never
  containers:
  - name: test
    image: busybox:1.28
    command: ['sh', '-c', 'echo "=== Dataset mount ==="; ls -la {DATA_DIR}/ 2>&1 | head -20; echo "=== Checkpoint mount ==="; ls -la {CHECKPOINT_DIR}/ 2>&1 | head -20; echo "=== DONE ==="']
    volumeMounts:
    - name: dataset
      mountPath: "{DATA_DIR}"
      readOnly: true
    - name: checkpoints
      mountPath: "{CHECKPOINT_DIR}"
  volumes:
  - name: dataset
    persistentVolumeClaim:
      claimName: blob-pvc-dataset
      readOnly: true
  - name: checkpoints
    persistentVolumeClaim:
      claimName: blob-pvc-checkpoint
"""

# Clean up previous test pod if exists
run('kubectl delete pod pvc-mount-test --ignore-not-found=true')
time.sleep(3)

# Create test pod
rc, out, err = run(f'echo {json.dumps(test_pod_yaml)} | kubectl apply -f -', shell=True)
# Write yaml to temp file and apply
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(test_pod_yaml)
    tmp_path = f.name

run('kubectl delete pod pvc-mount-test --ignore-not-found=true')
time.sleep(2)
rc, out, err = run(f'kubectl apply -f {tmp_path}')
if rc == 0:
    ok("Test pod created, waiting for completion...")
else:
    fail(f"Failed to create test pod: {err}")

# Wait for pod to complete
for i in range(30):
    rc, phase, _ = run('kubectl get pod pvc-mount-test -o jsonpath="{.status.phase}"')
    if phase in ['Succeeded', 'Failed']:
        break
    time.sleep(5)
    print(f"  Pod status: {phase} (attempt {i+1}/30)...")

# Get logs
rc, out, err = run('kubectl logs pvc-mount-test')
if rc == 0:
    if '=== DONE ===' in out:
        ok("Mount test completed successfully!")
    print("\n--- Test Pod Output ---")
    print(out)
else:
    fail(f"Could not get test pod logs: {err}")
    # Show pod events for debugging
    rc, events, _ = run('kubectl describe pod pvc-mount-test | findstr -i "warning\|error\|failed\|mount"')
    if events:
        print("\n--- Pod Events ---")
        print(events)

# Cleanup
run('kubectl delete pod pvc-mount-test --ignore-not-found=true')
os.unlink(tmp_path)

---
## 11. Summary Dashboard

In [ ]:
print("="*60)
print("           VALIDATION SUMMARY")
print("="*60)

checks = []

# Azure Auth
rc, _, _ = run('az account show -o none')
checks.append(('Azure Authentication', rc == 0))

# K8s
rc, _, _ = run('kubectl cluster-info 2>nul')
checks.append(('AKS Cluster Access', rc == 0))

# Storage Account
rc, _, _ = run(f'az storage account show -g {RG} -n {SA} -o none')
checks.append(('Storage Account', rc == 0))

# Blob Container
rc, out, _ = run(f'az storage container exists -n {CONTAINER} --account-name {SA} --query exists -o tsv')
checks.append(('Blob Container', rc == 0 and out.strip().lower() == 'true'))

# Blob CSI Driver
rc, out, _ = run(f'az aks show -g {RG} -n {AKS} --query "storageProfile.blobCsiDriver.enabled" -o tsv')
checks.append(('Blob CSI Driver', rc == 0 and out.strip().lower() == 'true'))

# StorageClasses
for sc in ['blobfuse-training-data', 'blobfuse-checkpoints']:
    rc, _, _ = run(f'kubectl get storageclass {sc} -o name 2>nul')
    checks.append((f'StorageClass: {sc}', rc == 0))

# PVCs
for pvc in ['blob-pvc-dataset', 'blob-pvc-checkpoint']:
    rc, _, _ = run(f'kubectl get pvc {pvc} -o name 2>nul')
    checks.append((f'PVC: {pvc}', rc == 0))

# Anyscale Operator
rc, out, _ = run('kubectl get pods -n anyscale-operator -o name 2>nul')
checks.append(('Anyscale Operator Pods', rc == 0 and len(out.strip()) > 0))

# Print results
passed = 0
for name, result in checks:
    icon = '\u2705' if result else '\u274C'
    print(f"  {icon}  {name}")
    if result:
        passed += 1

print(f"\n  Result: {passed}/{len(checks)} checks passed")
print("="*60)

if passed == len(checks):
    ok("ALL CHECKS PASSED — Ready to deploy!")
else:
    warn(f"{len(checks) - passed} check(s) need attention before deployment.")

---
## 12. Deploy StorageClasses & PVCs (Quick Deploy)
Run this if StorageClasses or PVCs are missing.

In [ ]:
# Deploy StorageClasses with substituted values
import re

sc_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'k8s', 'storageclass-blobfuse2.yaml')
with open(sc_path) as f:
    sc_yaml = f.read()

replacements = {
    '__STORAGE_ACCOUNT__': SA,
    '__CONTAINER__': CONTAINER,
    '__RESOURCE_GROUP__': RG,
    '__DATA_DIR__': DATA_DIR,
    '__CHECKPOINT_DIR__': CHECKPOINT_DIR,
    '__APP_DIR__': APP_DIR,
    '__CACHE_DIR__': CACHE_DIR,
    '__CHECKPOINT_CACHE__': CHECKPOINT_CACHE,
    '__AZURE_STORAGE_ACCOUNT_NAME__': SA,
    '__AZURE_STORAGE_ACCOUNT_KEY__': STORAGE_KEY,
}

for placeholder, value in replacements.items():
    sc_yaml = sc_yaml.replace(placeholder, value)

# Write to temp file and apply
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(sc_yaml)
    sc_tmp = f.name

rc, out, err = run(f'kubectl apply -f {sc_tmp}')
if rc == 0:
    ok(f"StorageClasses applied: {out}")
else:
    fail(f"Failed to apply StorageClasses: {err}")
os.unlink(sc_tmp)

# Apply PVCs
for pvc_file in ['k8s/pvc-training-data.yaml', 'k8s/pvc-checkpoint.yaml']:
    pvc_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), pvc_file)
    rc, out, err = run(f'kubectl apply -f {pvc_path}')
    if rc == 0:
        ok(f"PVC applied: {out}")
    else:
        fail(f"Failed to apply {pvc_file}: {err}")

print("\n--- Current PVCs ---")
rc, out, _ = run('kubectl get pvc')
print(out)

---
## 13. Deploy Ray HPO Job
Submit the CPU RayJob with all configuration substituted.

In [ ]:
# Create ConfigMap for application code
run('kubectl delete configmap hpo-app-code --ignore-not-found=true')
app_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'app')
rc, out, err = run(f'kubectl create configmap hpo-app-code --from-file={app_dir}')
if rc == 0:
    ok(f"ConfigMap created: {out}")
else:
    fail(f"Failed to create ConfigMap: {err}")

# Load and substitute RayJob YAML
rayjob_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'k8s', 'rayjob-cpu.yaml')
with open(rayjob_path) as f:
    rayjob_yaml = f.read()

rayjob_replacements = {
    '__DATA_DIR__': DATA_DIR,
    '__CHECKPOINT_DIR__': CHECKPOINT_DIR,
    '__APP_DIR__': APP_DIR,
    '__CACHE_DIR__': CACHE_DIR,
    '__CHECKPOINT_CACHE__': CHECKPOINT_CACHE,
    '__NUM_WORKERS__': NUM_WORKERS,
    '__WORKER_REPLICAS__': WORKER_REPLICAS,
}

for placeholder, value in rayjob_replacements.items():
    rayjob_yaml = rayjob_yaml.replace(placeholder, value)

# Write and apply
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(rayjob_yaml)
    rj_tmp = f.name

# Delete existing job first
run('kubectl delete rayjob hpo-job-cpu --ignore-not-found=true')
time.sleep(3)

rc, out, err = run(f'kubectl apply -f {rj_tmp}')
if rc == 0:
    ok(f"RayJob submitted: {out}")
else:
    fail(f"Failed to submit RayJob: {err}")
os.unlink(rj_tmp)

# Deploy Ray Dashboard service
dash_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'k8s', 'ray-dashboard-service.yaml')
if os.path.exists(dash_path):
    rc, out, _ = run(f'kubectl apply -f {dash_path}')
    ok(f"Dashboard service: {out}")

---
## 14. Monitor Job Status

In [ ]:
# Check RayJob status
rc, out, err = run('kubectl get rayjob -o wide')
if rc == 0:
    print("--- RayJobs ---")
    print(out)
else:
    warn(f"No RayJobs found: {err}")

# Check Ray pods
print("\n--- Ray Pods ---")
rc, out, _ = run('kubectl get pods -l ray.io/cluster -o wide')
if rc == 0:
    print(out)
else:
    rc, out, _ = run('kubectl get pods -o wide')
    print(out)

# Show events for any issues
print("\n--- Recent Events (last 5 min) ---")
rc, out, _ = run('kubectl get events --sort-by=.lastTimestamp --field-selector type!=Normal -o custom-columns="TIME:.lastTimestamp,TYPE:.type,REASON:.reason,MESSAGE:.message" 2>nul')
if rc == 0 and out.strip():
    # Show last 15 events
    lines = out.strip().split('\n')
    for line in lines[:16]:
        print(line)
else:
    ok("No warning/error events")